# Coffee17 preprocessing — Kaggle one-time OOF

Jalankan hanya setelah Validation Decision menghasilkan `AUTHORIZE_OOF_TEST_EVALUATION`. Tambahkan Coffee17 dataset dan output Validation Decision sebagai Kaggle Inputs. Inference only.


In [ ]:
CODE_COMMIT='7de2abb46efd5e71dbab508e2ae4b4e61102a7aa'
import hashlib, importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
PROJECT=WORK/'coffee17-preprocessing-project'; REPO=WORK/'coffee-bean-classification-code'

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()
def merge_tree_exact(source,target):
    for item in sorted(Path(source).rglob('*')):
        if not item.is_file(): continue
        dst=Path(target)/item.relative_to(source); dst.parent.mkdir(parents=True,exist_ok=True)
        if dst.is_file() and sha256_file(item)!=sha256_file(dst):
            raise RuntimeError(f'Input conflict: {item.relative_to(source)}')
        if not dst.exists(): shutil.copy2(item,dst)

prior=sorted(p for p in INPUT.rglob('coffee17-preprocessing-project') if p.is_dir())
if not prior: raise FileNotFoundError('Tambahkan output Validation Decision sebagai Kaggle Input.')
for p in prior: merge_tree_exact(p,PROJECT)

if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--quiet','--no-checkout','https://github.com/ediprin/coffee-bean-classification.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--quiet','--detach',CODE_COMMIT],check=True)
lock=PROJECT/'evidence/coffee17-preprocessing-runtime-v1/requirements_preprocessing_study_lock.txt'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(lock)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)

import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan Kaggle GPU untuk OOF inference.')

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.experiments.preprocessing_environment import verify_environment
from bilinear_lmmd.experiments.run_preprocessing_oof import run_oof
verify_environment(PROJECT/'evidence/coffee17-preprocessing-runtime-v1/runtime_environment.json')

by_class=discover_directory_samples(INPUT)
ARCHIVE=WORK/'coffee17_original.zip'
if ARCHIVE.exists(): ARCHIVE.unlink()
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info=zipfile.ZipInfo(f'{class_name}/{path.name}',date_time=(1980,1,1,0,0,0))
            info.compress_type=zipfile.ZIP_STORED; info.external_attr=0o644 << 16
            bundle.writestr(info,path.read_bytes())
CANONICAL=WORK/'coffee17_original_v1'; PROV=WORK/'coffee17_oof_provenance'
for p in (CANONICAL,PROV):
    if p.exists(): shutil.rmtree(p)
audit_coffee17_provenance(ARCHIVE,PROV,canonical_root=CANONICAL)

DATA=PROJECT/'evidence/coffee17-preprocessing-data-v1'
AUTH=PROJECT/'evidence/coffee17-preprocessing-primary-v1/preprocessing_primary_confirmation.json'
EXPERIMENTS=PROJECT/'experiments/coffee17-preprocessing-primary-v1'
OOF=PROJECT/'oof/coffee17-preprocessing-primary-v1'
result=run_oof(
    canonical_root=CANONICAL,
    clean_manifest=DATA/'clean_manifest.json',
    fold_manifest=DATA/'fold_manifest.json',
    authority_path=AUTH,
    experiments_root=EXPERIMENTS,
    output_root=OOF,
    authorize_test=True,
)
print('OOF COMPLETE:',result)
for p in (REPO,CANONICAL,PROV):
    if p.exists(): shutil.rmtree(p,ignore_errors=True)
if ARCHIVE.exists(): ARCHIVE.unlink()
print('Klik Save Version. Gunakan output ini sebagai input notebook Analysis.')
